In [175]:
import os
import rasterio
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, OPTICS, HDBSCAN
from sklearn.cluster import MiniBatchKMeans
from pyproj import Proj
from pyproj import Transformer
from pyproj import CRS
import subprocess

from sklearn.preprocessing import StandardScaler

In [230]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001_20240923_121008"
output_dir = os.path.join(rundir, "cluster_analysis")
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

n_patches = 3000
weights = [20,20,1,1,1] # xs, ys, slope, sin(aspect), cos(aspect)

In [231]:
def slope(infile, output_dir, working_dir = None):
        """
        Slope
        Description:
        Calculates a slope raster from an input DEM.
        Toolbox: Geomorphometric Analysis
        Parameters:

        Flag               Description
        -----------------  -----------
        -i, --dem          Input raster DEM file.
        -o, --output       Output raster file.
        --zfactor          Optional multiplier for when the vertical and horizontal units are not the same.
        --units            Units of output raster; options include 'degrees', 'radians', 'percent'


        Example usage:
        >>./whitebox_tools -r=Slope -v --wd="/path/to/data/" --dem=DEM.tif -o=output.tif --units="radians"
        Documentation: https://www.whiteboxgeo.com/manual/wbt_book/available_tools/geomorphometric_analysis.html#Slope
        """
        if working_dir is None: working_dir = output_dir
        outfile = os.path.join(output_dir, "slope.tif")
        completed_proc = subprocess.run(
            ["whitebox_tools",
            "-r=Slope",
            "--dem={}".format(infile),
            "-o={}".format(outfile),
            "--max_procs={}".format(8)],
            cwd=working_dir,
            stdout=subprocess.PIPE
        )
        completed_proc.check_returncode()  # raise CalledProcessError if return code is non-zero.
        return(outfile)

In [232]:
def aspect(infile, output_dir, working_dir = None):
    """
        Aspect
        Description:
        Calculates an aspect raster from an input DEM.
        Toolbox: Geomorphometric Analysis
        Parameters:

        Flag               Description
        -----------------  -----------
        -i, --dem          Input raster DEM file.
        -o, --output       Output raster file.
        --zfactor          Optional multiplier for when the vertical and horizontal units are not the same.


        Example usage:
        >>./whitebox_tools -r=Aspect -v --wd="/path/to/data/" --dem=DEM.tif -o=output.tif
    """
    if working_dir is None: working_dir = output_dir
    outfile = os.path.join(output_dir, "aspect.tif")
    completed_proc = subprocess.run(
        ["whitebox_tools",
        "-r=Aspect",
        "--dem={}".format(infile),
        "-o={}".format(outfile),
        "--max_procs={}".format(8)],
        cwd=working_dir,
        stdout=subprocess.PIPE
    )
    completed_proc.check_returncode()  # raise CalledProcessError if return code is non-zero.
    return(outfile)

In [ ]:
bathyfile = os.path.join(rundir, "bathy.tif")
slope(bathyfile, output_dir=output_dir)
aspect(bathyfile, output_dir=output_dir)

In [ ]:
for filename in filter(lambda file: file.endswith(".tif"), os.listdir(output_dir)): 
    with rasterio.open(os.path.join(output_dir,filename)) as raster:
        print(raster.name)

In [235]:
def write_tif(fname, data, profile):
    "Write .tif data and profile using rasterio."
    print(f"Write file: {fname}")
    with rasterio.open(fname, 'w', **profile) as dst:
        dst.write(data, 1)

In [ ]:
# Load files
from pyproj import CRS
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info

with rasterio.open(os.path.join(output_dir, "slope.tif")) as ds:
    slope, slope_profile = ds.read(1), ds.profile.copy()

with rasterio.open(os.path.join(output_dir, "aspect.tif")) as ds:
    aspect, aspect_profile = ds.read(1), ds.profile.copy()
    sin_aspect = np.sin(aspect*np.pi/180.)
    cos_aspect = np.cos(aspect*np.pi/180.)

with rasterio.open(os.path.join(rundir, "bathy.tif")) as bathy_ds:
    z = bathy_ds.read(1)
    bathy_transform = bathy_ds.transform
    
    cols, rows = np.meshgrid(np.arange(bathy_ds.width), np.arange(bathy_ds.height))
    xs_grid, ys_grid = bathy_ds.transform*(cols, rows)
    
    if bathy_ds.crs.to_epsg() == 4326:
        assert bathy_ds.crs.units_factor[0] == "degree"
        # lonlat coordinates. Find suitable utm coordinates to calculate area
        # left, bottom, right, top = bathy_ds.bounds
        utm_crs_list = query_utm_crs_info(
            datum_name="WGS 84",
            area_of_interest=AreaOfInterest(
                west_lon_degree=bathy_ds.bounds.left,
                south_lat_degree=bathy_ds.bounds.bottom,
                east_lon_degree=bathy_ds.bounds.right,
                north_lat_degree=bathy_ds.bounds.top,
            ),
        )
        utm_crs = CRS.from_epsg(utm_crs_list[0].code)
        print(utm_crs.to_wkt())
        transformer = Transformer.from_crs(bathy_ds.crs.to_proj4(), utm_crs)
        
        utm_xs_grid, utm_ys_grid = transformer.transform(xs_grid, ys_grid)
        dx, dy = np.pad(np.diff(utm_xs_grid),((0,0),(0,1)), 'edge'), np.pad(np.diff(utm_ys_grid, axis=0),((0,1),(0,0)), 'edge')
        
        #upper_left, lower_right = [transformer.transform(xs_grid[col,row], ys_grid[col,row]) for col, row in [(1000,1000), (1001,1001)]]
        #dx, dy = np.abs(np.array(upper_left) - np.array(lower_right))
    else:
        assert bathy_ds.crs.units_factor[0] in ["m", "metre"]
        dx, dy = np.pad(np.diff(xs_grid),((0,0),(0,1)), 'edge'), np.pad(np.diff(ys_grid, axis=0),((0,1),(0,0)), 'edge')
    area = np.abs(dx*dy)


mask = z<0
df = pd.DataFrame(np.vstack([xs_grid[mask], ys_grid[mask], z[mask], slope[mask], sin_aspect[mask], cos_aspect[mask], area[mask]]).T,
                  columns=["xs", "ys", "z", "slope", "sin_aspect", "cos_aspect", "area"])
                  

In [ ]:
df

In [238]:
X = df[["xs", "ys", "slope", "sin_aspect", "cos_aspect"]].values
scaler = StandardScaler()
scaler.fit(X)
X_trans = np.matmul(scaler.transform(X), np.diag(weights))

In [ ]:
# Fit the labels
#kmeans = KMeans(n_clusters=200, random_state=0, n_init="auto", algorithm="lloyd")
kmeans = MiniBatchKMeans(n_clusters=n_patches, max_no_improvement=20, batch_size=int(256*n_patches/100)) # This is much faster
kmeans.fit(X_trans)

In [240]:
# Get the labels
patches = np.empty(bathy_ds.shape)
patches[:] = np.nan
patches[mask] = kmeans.labels_

df["patch"] = kmeans.labels_

In [ ]:
write_tif(os.path.join(output_dir, f"patches_{n_patches}.tif"), patches, slope_profile)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(patches, cmap="tab20b")

In [ ]:
df

In [244]:
# Calculate areas
areas = df.groupby("patch")[["area"]].sum()

In [ ]:
# Distribution of areas for each patch
plt.hist(areas, bins=30)

## Plot statistical features of patches

In [246]:
sin_aspect_by_patch = {f"patch_{i}": df[df.patch == i].sin_aspect.values for i in range(200)}
cos_aspect_by_patch = {f"patch_{i}": df[df.patch == i].cos_aspect.values for i in range(200)}
slope_by_patch = {f"patch_{i}": df[df.patch == i].slope.values for i in range(200)}

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, layout='constrained')
fig.set_size_inches(18, 8)

patches = slice(0,50)

ax1.boxplot(list(slope_by_patch.values())[patches], showfliers=False)
ax1.set_label("Slope")

ax2.boxplot(list(sin_aspect_by_patch.values())[patches], showfliers=False)
ax2.set_label("sin(aspect)")

ax3.boxplot(list(cos_aspect_by_patch.values())[patches], showfliers=False)
ax3.set_label("cos(aspect)")

plt.show()

# Combining patches into larger volumes.

We are interested in applying the partitioning to select potential volumes.

In [ ]:
centers_df = pd.DataFrame(kmeans.cluster_centers_, columns=["xs", "ys", "slope", "sin_aspect", "cos_aspect"])
centers_df

In [184]:
# Cluster centers using KMeans

volume_refinements = [1000, 500, 200]

def cluster_patches(n_clusters, patch_centers):
    kmeans_centers = MiniBatchKMeans(n_clusters=n_clusters, max_no_improvement=20)
    kmeans_centers.fit(patch_centers)
    return(kmeans_centers.labels_)

for n_clusters in volume_refinements:
    labels = cluster_patches(n_clusters, kmeans.cluster_centers_)
    df[f"Volume_{n_clusters}"] = df.apply(lambda row: labels[int(row["patch"])], axis=1)

In [ ]:
df

In [186]:
# Find all patch numbers with a given center label
volumes = {}
for n_clusters in volume_refinements:
    volume = np.empty(bathy_ds.shape)
    volume[:] = np.nan
    df[f"volume_{n_clusters}"] = np.nan
    volume[mask] = df[f"Volume_{n_clusters}"].values
    volumes[n_clusters] = volume

In [ ]:
for i, n_clusters in enumerate(volume_refinements):
    write_tif(os.path.join(output_dir, f"patches_{i + 1}.tif"), volumes[n_clusters], slope_profile)

In [249]:
# Cluster centers using DBSCAN

minmax_cluster_size = [(2,3), (3,5), (5,30)]

def cluster_patches(min_cluster_size, max_cluster_size, patch_centers):
    hdbscan_centers = HDBSCAN(min_cluster_size=min_cluster_size, max_cluster_size=max_cluster_size, leaf_size=40)
    hdbscan_centers.fit(patch_centers)
    return(hdbscan_centers.labels_)

for min_cluster_size, max_cluster_size in minmax_cluster_size:
    labels = cluster_patches(min_cluster_size, max_cluster_size, kmeans.cluster_centers_)
    df[f"Volume_{max_cluster_size}"] = df.apply(lambda row: labels[int(row["patch"])], axis=1)

In [250]:
# Find all patch numbers with a given center label
volumes = {}
for _, max_cluster_size in minmax_cluster_size:
    volume = np.empty(bathy_ds.shape)
    volume[:] = np.nan
    df[f"volume_{max_cluster_size}"] = np.nan
    volume[mask] = df[f"Volume_{max_cluster_size}"].values
    volumes[max_cluster_size] = volume

In [ ]:
for i,(_,max_cluster_size) in enumerate(minmax_cluster_size):
    write_tif(os.path.join(output_dir, f"patches_{i+1}.tif"), volumes[max_cluster_size], slope_profile)